# Observatorio de Obras Públicas · 01 Extracción (SECOP I + SECOP II)

Trae los contratos de **obra** reales de SECOP I y SECOP II, los une en una sola tabla sin duplicados y la guarda en `datos/obras.csv`.

También guarda en `datos/sin_contrato.csv` los procesos de obra que en SECOP I figuran **adjudicados pero sin contrato publicado** (no tienen fecha de firma, contratista ni valor del contrato). No se suman a la inversión, pero el tablero los muestra aparte porque cambian la comparación entre municipios.

**Para analizar otra región, solo cambia la celda 1.**

## 1 · Configuración (lo único que se cambia)

In [1]:
import datetime

DEPARTAMENTO = "CAUCA"
MUNICIPIOS   = ["GUAPI", "TIMBIQUI", "LOPEZ DE MICAY"]   # sin tildes, en mayúsculas
ANIO_INICIO  = 2022
ANIO_FIN     = datetime.date.today().year   # siempre hasta el año actual

## 2 · Herramientas

In [2]:
%pip install -q duckdb

import os, unicodedata
import requests
import pandas as pd
import duckdb

FUENTES = {
    "SECOP II": "https://www.datos.gov.co/resource/jbjy-vk9h.json",
    "SECOP I":  "https://www.datos.gov.co/resource/f789-7hwg.json",
}
pd.options.display.float_format = "{:,.0f}".format
print("Listo.")

Note: you may need to restart the kernel to use updated packages.


Listo.


## 3 · Funciones
SECOP I y SECOP II llaman distinto a las mismas columnas. `CAMPOS` lista los nombres posibles de cada dato y el código usa el que exista en cada fuente.

Algunas entidades aparecen en SECOP II con el departamento como *No Definido* (por ejemplo, "MUNICIPIO DE GUAPI CAUCA"). Por eso el filtro de la API acepta el departamento **o** el nombre de la entidad, y después `resolver_municipio` deja solo los municipios elegidos.

In [3]:
CAMPOS = {
    "id":          ["id_contrato", "uid", "numero_de_constancia", "numero_del_contrato"],
    "departamento":["departamento", "departamento_entidad", "departamento_ejecucion"],
    "municipio":   ["ciudad", "municipio_entidad", "municipio_ejecucion", "municipios_ejecucion", "municipio_de_entrega"],
    "tipo":        ["tipo_de_contrato", "tipo_contrato"],
    "fecha_firma": ["fecha_de_firma", "fecha_de_firma_del_contrato", "fecha_firma"],
    "objeto":      ["objeto_del_contrato", "descripcion_del_proceso", "detalle_del_objeto_a_contratar", "objeto_a_contratar"],
    "valor":       ["valor_del_contrato", "cuantia_contrato", "valor_contrato"],
    "valor_con_adiciones": ["valor_contrato_con_adiciones"],
    "entidad":     ["nombre_entidad", "entidad", "nombre_de_la_entidad"],
    "contratista": ["proveedor_adjudicado", "nom_razon_social_contratista"],
    "fecha_inicio":["fecha_de_inicio_del_contrato", "fecha_ini_ejec_contrato"],
    "fecha_fin":   ["fecha_de_fin_del_contrato", "fecha_fin_ejec_contrato"],
    "dias_adicionados": ["dias_adicionados", "tiempo_adiciones_en_dias"],
    "estado":      ["estado_contrato", "estado_del_proceso"],
    "modalidad":   ["modalidad_de_contratacion"],
    "url":         ["urlproceso", "ruta_proceso_en_secop_i"],
}


def normalizar(texto):
    """Quita tildes, espacios sobrantes y pasa a mayúsculas."""
    if texto is None or (isinstance(texto, float) and pd.isna(texto)):
        return ""
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.upper().split())


def columnas_de(url):
    """Nombres de las columnas reales de una fuente."""
    r = requests.get(url, params={"$limit": 1}, timeout=60)
    datos = r.json()
    if r.status_code != 200 or not isinstance(datos, list) or not datos:
        raise RuntimeError(f"La fuente {url} respondió {r.status_code}: {str(datos)[:300]}")
    return list(datos[0].keys())


def elegir(columnas):
    """Para cada dato de CAMPOS, la primera columna que exista en la fuente."""
    return {dato: next((c for c in opciones if c in columnas), None) for dato, opciones in CAMPOS.items()}


def consultar_api(url, where, tamano_pagina=5000):
    """Trae todos los registros que cumplen el filtro, por páginas."""
    filas, offset = [], 0
    while True:
        r = requests.get(url, params={"$where": where, "$limit": tamano_pagina,
                                      "$offset": offset, "$order": ":id"}, timeout=180)
        if r.status_code != 200:
            raise RuntimeError(f"La API respondió {r.status_code}: {r.text[:400]}")
        pagina = r.json()
        filas.extend(pagina)
        print(f"    {len(filas):,} registros...")
        if len(pagina) < tamano_pagina:
            return pd.DataFrame(filas)
        offset += tamano_pagina


def filtro_lugar(col):
    """El departamento, o el nombre de la entidad cuando SECOP deja el departamento en 'No Definido'."""
    partes = [f"upper({col['departamento']}) like '%{DEPARTAMENTO}%'"]
    if col["entidad"]:
        # Palabra más larga del municipio sin su última letra, para no depender de tildes: 'TIMBIQU' encuentra 'Timbiquí'
        partes += [f"upper({col['entidad']}) like '%{max(m.split(), key=len)[:-1]}%'" for m in MUNICIPIOS]
    return "(" + " OR ".join(partes) + ")"


def resolver_municipio(ciudad, entidad, objetivo):
    """Usa ciudad y respalda con el nombre de entidad cuando SECOP abrevia el lugar."""
    ciudad = normalizar(ciudad)
    entidad = normalizar(entidad)
    for municipio in objetivo:
        if municipio in ciudad:
            return municipio
    for municipio in objetivo:
        if municipio in entidad:
            return municipio
    return ""


def traer_fuente(nombre, url):
    print(f"\n{nombre}")
    col = elegir(columnas_de(url))
    faltan = [d for d in ("id", "departamento", "tipo", "fecha_firma") if not col[d]]
    if not col["municipio"] and not col["entidad"]:
        faltan.append("municipio/entidad")
    if faltan:
        raise RuntimeError(f"{nombre}: faltan columnas requeridas {faltan}; columnas detectadas: {col}")

    where = f"{filtro_lugar(col)} AND upper({col['tipo']}) like '%OBRA%'"
    try:
        df = consultar_api(url, where + f" AND {col['fecha_firma']} >= '{ANIO_INICIO}-01-01T00:00:00'")
    except RuntimeError as error:
        print(f"  La fecha no se pudo filtrar en la API ({error}); se filtra después.")
        df = consultar_api(url, where)
    if df.empty:
        raise RuntimeError(f"{nombre}: la fuente no devolvió contratos tipo Obra para {DEPARTAMENTO}")

    t = pd.DataFrame({dato: df[c] if c and c in df else None for dato, c in col.items()})
    t["fuente"] = nombre
    objetivo = [normalizar(m) for m in MUNICIPIOS]
    t["municipio"] = t.apply(lambda fila: resolver_municipio(fila["municipio"], fila["entidad"], objetivo), axis=1)
    t["fecha_firma"] = pd.to_datetime(t["fecha_firma"], errors="coerce")
    for c in ("valor", "valor_con_adiciones", "dias_adicionados"):
        t[c] = pd.to_numeric(t[c], errors="coerce")

    t = t[t["municipio"].isin(objetivo)]
    t = t[t["fecha_firma"].dt.year.between(ANIO_INICIO, ANIO_FIN)]
    if t.empty:
        raise RuntimeError(f"{nombre}: no hay contratos de los municipios seleccionados tras aplicar ciudad/entidad y fechas")
    print(f"  -> {len(t):,} contratos de obra en los municipios elegidos")
    return t


def traer_sin_contrato(url):
    """SECOP I: procesos de obra adjudicados que la entidad nunca registró como contrato (sin fecha de firma)."""
    print("\nSECOP I · adjudicados sin contrato publicado")
    where = (f"upper(departamento_entidad) like '%{DEPARTAMENTO}%' AND upper(tipo_de_contrato) like '%OBRA%' "
             f"AND estado_del_proceso = 'Adjudicado' AND fecha_de_firma_del_contrato IS NULL "
             f"AND fecha_de_cargue_en_el_secop >= '{ANIO_INICIO}-01-01T00:00:00'")
    df = consultar_api(url, where)
    if df.empty:
        return pd.DataFrame(columns=["id", "proceso", "fuente", "municipio", "entidad", "fecha_cargue",
                                     "objeto", "presupuesto", "modalidad", "url"])
    t = pd.DataFrame({
        "id": df["uid"],
        "proceso": df.get("numero_de_proceso"),
        "fuente": "SECOP I",
        "municipio": df.get("municipio_entidad"),
        "entidad": df.get("nombre_entidad"),
        "fecha_cargue": pd.to_datetime(df.get("fecha_de_cargue_en_el_secop"), errors="coerce"),
        "objeto": df.get("detalle_del_objeto_a_contratar"),
        "presupuesto": pd.to_numeric(df.get("cuantia_proceso"), errors="coerce"),
        "modalidad": df.get("modalidad_de_contratacion"),
        "url": df.get("ruta_proceso_en_secop_i"),
    })
    objetivo = [normalizar(m) for m in MUNICIPIOS]
    t["municipio"] = t.apply(lambda fila: resolver_municipio(fila["municipio"], fila["entidad"], objetivo), axis=1)
    t = t[t["municipio"].isin(objetivo) & t["fecha_cargue"].dt.year.between(ANIO_INICIO, ANIO_FIN)]
    # Un mismo proceso puede venir en dos filas; se deja una
    t = t[~(t["proceso"].notna() & t["proceso"].duplicated())].reset_index(drop=True)
    print(f"  -> {len(t):,} procesos adjudicados sin contrato publicado")
    return t

## 4 · Traer y unir SECOP I y SECOP II

In [4]:
tablas = [traer_fuente(n, u) for n, u in FUENTES.items()]
obras = pd.concat([t for t in tablas if not t.empty], ignore_index=True)

antes = len(obras)
obras = obras.drop_duplicates(subset=["id", "fuente"]).reset_index(drop=True)
print(f"\nTotal: {len(obras):,} contratos ({antes - len(obras)} duplicados eliminados)")
print(obras.groupby(["fuente", "municipio"]).size())

sin_contrato = traer_sin_contrato(FUENTES["SECOP I"])
if not sin_contrato.empty:
    print(sin_contrato.groupby("municipio")["presupuesto"].agg(["size", "sum"]))


SECOP II


    3,259 registros...
  -> 6 contratos de obra en los municipios elegidos

SECOP I


    3,932 registros...
  -> 360 contratos de obra en los municipios elegidos

Total: 364 contratos (2 duplicados eliminados)
fuente    municipio     
SECOP I   GUAPI              72
          LOPEZ DE MICAY    276
          TIMBIQUI           10
SECOP II  GUAPI               1
          TIMBIQUI            5
dtype: int64

SECOP I · adjudicados sin contrato publicado


    259 registros...
  -> 110 procesos adjudicados sin contrato publicado
                size          sum
municipio                        
GUAPI             11   3510225190
LOPEZ DE MICAY    47  21027086017
TIMBIQUI          52  31177076336


## 5 · Primera consulta SQL: obras y valor por municipio y año
Primer vistazo. Todavía falta la limpieza (paso 02).

In [5]:
duckdb.sql("""
    SELECT municipio,
           year(fecha_firma)  AS anio,
           COUNT(*)           AS obras,
           SUM(valor)         AS valor_contratado
    FROM obras
    GROUP BY municipio, anio
    ORDER BY municipio, anio
""").df()

,municipio,anio,obras,valor_contratado
0,GUAPI,2022,5,"7,564,121,120"
1,GUAPI,2023,2,"543,546,378"
2,GUAPI,2024,15,"813,420,312"
3,GUAPI,2025,31,"7,145,046,246"
4,GUAPI,2026,20,"12,045,714,838"
5,LOPEZ DE MICAY,2022,85,"14,240,683,314"
6,LOPEZ DE MICAY,2023,63,"1,953,458,447"
7,LOPEZ DE MICAY,2024,78,"3,203,837,143"
8,LOPEZ DE MICAY,2025,37,"7,542,614,847"
9,LOPEZ DE MICAY,2026,13,"2,676,137,031"


## 6 · Guardar

In [6]:
os.makedirs("datos", exist_ok=True)
obras.to_csv("datos/obras.csv", index=False)
sin_contrato.to_csv("datos/sin_contrato.csv", index=False)
print("Guardado: datos/obras.csv y datos/sin_contrato.csv")

try:   # si el código corre en un servidor de Colab, descarga los archivos
    from google.colab import files
    files.download("datos/obras.csv")
    files.download("datos/sin_contrato.csv")
except Exception:
    pass

Guardado: datos/obras.csv y datos/sin_contrato.csv
